# Truflation EV Fiat Market - Report of Risk Parameters

In [59]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import get_risk_parameters

Set data-specific parameters:

In [60]:
filename = "btc_ev_v12_fiat"

path_to_file = os.path.join(os.getcwd().split("params")[0], f"data/{filename}")

periodicity = 86400  # 1 day in seconds
cap = 10  # cap on pay off, set by governance

short_twap = 2 * periodicity
long_twap = 10 * periodicity

periodicity, short_twap, long_twap

(86400, 172800, 864000)

## Understanding the Data

Load and plot the data:

In [61]:
df = pd.read_csv(path_to_file+".csv").set_index("date")
df[["close", "twap"]].plot()

# Computing Risk Parameters

In [62]:
#get_risk_parameters.main(
#    path_to_file, periodicity, cap, short_twap, long_twap
#)

In [63]:
RESULTS_FOLDER = os.path.join(os.getcwd(), f"cache/{filename}")
RESULTS_FOLDER, os.path.isdir(RESULTS_FOLDER)

('/Users/fredericoteixeira/Projects/overlay/overlay-risk/params/cache/btc_ev_v12_fiat',
 True)

## Funding Rate - `k`

Compute `k`, the funding related risk metric.

It means that with this `k`, if OI exists only on one side (the worst case scenario), then in $1-\alpha$% of cases, the OI will get drawn down to zero just due to funding over the course of the next `n` days.

In [64]:
df_ks = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "btc_ev_v12_fiat-ks.csv")
).rename(columns={"Unnamed: 0": "index"})

df_ks.index = np.array([float(idx.split("=")[1])/periodicity for idx in df_ks["index"]])
df_ks.index.name = "days"
df_ks.drop("index", axis=1, inplace=True)

df_ks.columns = [float(c.split("=")[1]) for c in df_ks.columns]
df_ks.columns.name = "alpha"

fig = df_ks.plot(
    title=f"Truflation EF Fiat - funding rate - cap: {cap}, short twap: {short_twap}, long twap: {long_twap}"
)
fig.show()
fig.write_image(
    os.path.join(RESULTS_FOLDER, f"btc_ev_v12_fiat-ks-cp-{cap}-st-{short_twap}-lt-{long_twap}.png")
)

### Recommendation

Anchor time = 45 days (3888000 secs), Confidence = 95% (alpha=0.05)

In [65]:
int(df_ks.loc[45, 0.05] * 1e18)

148560720450

## Spread - $\delta$

In [66]:
df_delta = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "btc_ev_v12_fiat-deltas.csv")
).set_index("alpha", drop=True)

fig = df_delta.plot(
    title=f"Truflation EF Fiat - spread - cap: {cap}, short twap: {short_twap}, long twap: {long_twap}"
)
fig.show()
fig.write_image(
    os.path.join(RESULTS_FOLDER, f"btc_ev_v12_fiat-deltas-cp-{cap}-st-{short_twap}-lt-{long_twap}.png")
)


### Recommendation

Confidence = 95% ($\alpha=0.05$)

In [67]:
int(df_delta.loc[0.05] * 1e18)

18976000843077100

## Impact - $\lambda$

In [68]:
df_lambda = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "btc_ev_v12_fiat-lambdas.csv")
)

df_lambda.index = [
    float(idx.split("=")[1]) for idx in df_lambda["Unnamed: 0"]
]
df_lambda.index.name = "alpha"
df_lambda.drop("Unnamed: 0", axis=1, inplace=True)

df_lambda.columns = [
    float(q.split("=")[1]) for q in df_lambda.columns
]
df_lambda.columns.name = "rolling volume"

fig = df_lambda.plot(
    title=f"Truflation EF Fiat - Impact - cap: {cap}, short twap: {short_twap}, long twap: {long_twap}"
)
fig.show()
fig.write_image(
    os.path.join(RESULTS_FOLDER, f"btc_ev_v12_fiat-lambdas-cp-{cap}-st-{short_twap}-lt-{long_twap}.png")
)

### Recommendation

Confidence = 95% ($\alpha=0.05$); rolling volume = 1% of cap (rolling volume $= 0.01$)

In [69]:
int(df_lambda.loc[0.05, 0.01] * 1e18)

24827178309162258432

## Liquidations - `maintenanceMarginFraction`

In [70]:
df_mmf = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "btc_ev_v12_fiat-mms.csv")
)

df_mmf.index = [
    int(idx.split("=")[1]) for idx in df_mmf["Unnamed: 0"]
]
df_mmf.index.name = "seconds"
df_mmf.drop(columns=["Unnamed: 0"], inplace=True)

df_mmf.columns = [
    float(c.split("=")[1]) for c in df_mmf.columns
]
df_mmf.columns.name = "alpha"

fig = df_mmf.plot(
    title=f"Truflation EF Fiat - maintenance margin fraction - cap: {cap}, short twap: {short_twap}, long twap: {long_twap}"
)
fig.show()
fig.write_image(
    os.path.join(RESULTS_FOLDER, f"btc_ev_v12_fiat-mmf-cp-{cap}-st-{short_twap}-lt-{long_twap}.png")
)


### Recommendation

Time taken for position to go negative = 4 hours; Confidence = 95%

In [71]:
int(df_mmf.loc[14400, 0.05] * 1e18)

2112740904102399

## Liquidations - `maintenanceMarginBurnRate`

In [72]:
df_betas = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "btc_ev_v12_fiat-betas.csv")
)

df_betas.index = [
    int(idx.split("=")[1]) for idx in df_betas["Unnamed: 0"]
]
df_betas.index.name = "seconds"
df_betas.drop(columns=["Unnamed: 0"], inplace=True)

df_betas.columns = [
    float(c.split("=")[1]) for c in df_betas.columns
]
df_betas.columns.name = "alpha"

fig = df_betas.plot(
    title=f"Truflation EF Fiat - Maintenance margin burn rate - cap: {cap}, short twap: {short_twap}, long twap: {long_twap}"
)
fig.show()
fig.write_image(
    os.path.join(RESULTS_FOLDER, f"btc_ev_v12_fiat-betas-cp-{cap}-st-{short_twap}-lt-{long_twap}.png")
)

### Recommendation

Time taken for position to go negative = 4 hours (`index=14400`); Confidence = 95%  ($\alpha=0.05$).

In [73]:
int(df_betas.loc[14400, 0.05] * 1e18)

441082356105995264

## Price drift - `priceDriftUpperLimit`

In [74]:
df_mus = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "btc_ev_v12_fiat-mu_maxs.csv")
).set_index("alpha", drop=True)

fig = df_mus.plot(
    title=f"Truflation EF Fiat - Price drift - cap: {cap}, short twap: {short_twap}, long twap: {long_twap}"
)
fig.show()
fig.write_image(
    os.path.join(RESULTS_FOLDER, f"btc_ev_v12_fiat-mu_maxs-cp-{cap}-st-{short_twap}-lt-{long_twap}.png")
)

### Recommendation

Confidence = 99.9% ($\alpha=0.01$, according to the whitepaper)

In [75]:
mu_max = df_mus.loc[0.01].values[0]
print(int(mu_max * 1e18))

3573906678993


Furthermore, $\mu_{max}=2.4852 \times 10^{-6}$ means if price goes up/down `23.95%` in a duration equal to the longer TWAP (ie, 1 day), then the market doesn’t honor this spiked price. 

Remark: Equation (58) on page 10 of the whitepaper describes the formula for getting the percentage $e^{86400 \times \mu_{max}}$.

In [76]:
print("Price drift of ", mu_max, " ==> " , round(100 * (np.exp(periodicity * mu_max) - 1), 2), "%")

Price drift of  3.573906678993899e-06  ==>  36.18 %


If $\alpha = 0.05$, then the percentage comes `3.32 %` which is very small and pretty likely to get hit.

In [77]:
round(100 * (np.exp(periodicity * df_mus.loc[0.05].values[0])-1), 2)

4.79